In [124]:
from dotenv import load_dotenv
load_dotenv()
from tqdm import tqdm

In [138]:
TARGET_PROMPT_ID = 8

In [139]:
import polars as pl
from utils import load_asap_dataset, get_score_range
from sklearn.model_selection import train_test_split

asap = load_asap_dataset("./dataset", essay_set=TARGET_PROMPT_ID)
train_df, test_df = train_test_split(asap, test_size=0.1, random_state=12)
train_df = train_df.sample(30, shuffle=True)

print(f"train len: {len(train_df)}")
print(f"test len: {len(test_df)}")

train len: 30
test len: 73


In [140]:
train_df.sort(by='score')

essay_set,essay_id,essay,score
i64,i64,str,i64
8,21240,"""I think that laughter is a ver…",26
8,21448,""" A time when laughter was one …",30
8,21178,""" @CAPS1 is laughter important?…",31
8,21544,""" I choose to have a positive o…",31
8,21256,""" Boy says too @CAPS2:@CAPS1 we…",33
…,…,…,…
8,20884,""" Laughter brings the world tog…",43
8,21059,""" Laughter is one of the most i…",45
8,21385,""" All of the memories that I've…",50


In [141]:
test_df

essay_set,essay_id,essay,score
i64,i64,str,i64
8,20838,"""Iv always loved laughter in my…",27
8,20728,""" Starting a story out with two…",44
8,20972,""" Falling in love, @CAPS…",32
8,21308,"""During the @DATE1 I decided to…",40
8,21298,""" One stormy winter night my g…",30
…,…,…,…
8,21029,""" the @CAPS1 @CAPS2 trip …",30
8,21532,""" ""@CAPS1 days are diamonds and…",50
8,20806,""" There are a couple things tha…",35


In [142]:
# Create LLM queries
minscore, maxscore = get_score_range("ASAP", TARGET_PROMPT_ID)
queries = {}

many_shot_samples = ""
for essay, score in train_df.sort(by='score').select(['essay', 'score']).iter_rows():
    sample = f"score: {score}\nessay text: {essay}\n"
    many_shot_samples += sample

with open("./llm_prompts/many_shot_system.md", 'r') as f:
    system = f.read()

with open("./llm_prompts/many_shot_user.md", 'r') as f:
    user = f.read()

with open(f"llm_prompts/asap_original/prompt_{TARGET_PROMPT_ID}.md", "r") as f:
    prompt = f.read()

for id, essay in tqdm(test_df.sort(by='essay_id').select(['essay_id', 'essay']).iter_rows()):
    system_message = (
        system
            .replace('{prompt}', prompt)
            .replace('{examples}', many_shot_samples)
            .replace('{minscore}', str(minscore))
            .replace('{maxscore}', str(maxscore))
    )

    user_message = user.replace("{essay}", essay)

    queries[id] = [
        {'role': 'system', 'content': system_message},
        {'role': 'system', 'content': user_message}
    ]

73it [00:00, 4301.19it/s]


In [143]:
import tiktoken


def count_total_tokens(queries, model="gpt-4o-mini"):
    enc = tiktoken.encoding_for_model(model)
    total_tokens = 0

    for qlist in queries.values():
        for msg in qlist:
            total_tokens += len(enc.encode(msg["content"]))

    return total_tokens


total = count_total_tokens(queries)
print(f"総トークン数: {total:,}")

総トークン数: 1,612,974


In [144]:
from pathlib import Path

# MODEL = "gpt-5.2-2025-12-11"
MODEL = "gpt-4o-2024-11-20"
MAX_WORKERS = 20
MAX_RETRIES = 6
BASE_SLEEP = 1.0  # 初回スリープ秒
OUTDIR = Path(f"out/many_shot/{MODEL}/{TARGET_PROMPT_ID}")
JSONL_PATH = Path(f"out/many_shot/{MODEL}/{TARGET_PROMPT_ID}/results.jsonl")

In [145]:
import os
import json
import time
import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Dict, Any

from tqdm import tqdm
from openai import OpenAI
from openai import APIError, RateLimitError, InternalServerError, APITimeoutError

# ===================== ユーザー入力想定 =====================
# queries: {id: messages} という dict を想定（質問のコードのままでOK）
# 例: messages は Responses API の input 互換（文字列 or messages配列）
# queries = {...}
# =========================================================

client = OpenAI()  # OPENAI_API_KEY は環境変数から

# JSONL_PATH は既存コードの定義をそのまま使う想定
# 例:
# from pathlib import Path
# JSONL_PATH = Path("out.jsonl")

JSONL_PATH.parent.mkdir(parents=True, exist_ok=True)
JSONL_PATH.touch(exist_ok=True)


def load_done_ids() -> set:
    """JSONL に記録済みの id を既読扱いにする。"""
    done = set()
    with JSONL_PATH.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                obj = json.loads(line)
                if "id" in obj:
                    done.add(str(obj["id"]))
            except json.JSONDecodeError:
                pass
    return done


def dump_jsonl(record: Dict[str, Any]) -> None:
    """1レコードを JSONL に追記（都度 flush）。"""
    line = json.dumps(record, ensure_ascii=False)
    with JSONL_PATH.open("a", encoding="utf-8") as f:
        f.write(line + "\n")
        f.flush()


def call_api_once(id_: str, messages):
    # response = client.responses.create(
    #     model=MODEL,
    #     input=messages,
    #     reasoning={
    #         "effort": "medium"
    #     },
    #     text={
    #         "verbosity": "medium"
    #     }
    # )

    response = client.responses.create(
        model=MODEL,
        max_output_tokens=512,
        temperature=0.1,
        input=messages,
    )

    return response


def call_with_retry(id_: str, messages):
    # 指数バックオフ＋ジッター
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return call_api_once(id_, messages)
        except (RateLimitError, APITimeoutError, InternalServerError, APIError):
            if attempt == MAX_RETRIES:
                raise
            sleep = BASE_SLEEP * (2 ** (attempt - 1))
            sleep = sleep * (0.5 + random.random())  # ジッター
            time.sleep(sleep)
        except Exception:
            raise


def worker(id_: str, messages):
    resp = call_with_retry(id_, messages)

    # 便利プロパティ（全テキスト統合）
    try:
        text = resp.output_text
    except Exception:
        text = None

    # Count Cached Tokens
    # SDKの返り値が「属性アクセス」できる場合
    try:
        cached = resp.usage.input_tokens_details.cached_tokens
    except Exception:
        # 「辞書アクセス」になる環境向け（安全策）
        cached = resp["usage"]["input_tokens_details"]["cached_tokens"]

    # JSONL に追記（到着順ログ）
    record = {
        "id": str(id_),
        "text": text,
        "saved_at": int(time.time()),
        "cached_token": cached,
        "model": MODEL,
    }
    dump_jsonl(record)

    return id_, text


def run_all(queries: Dict[str, Any]):
    done_ids = load_done_ids()
    todo = [(str(i), m) for i, m in queries.items() if str(i) not in done_ids]

    if not todo:
        print("すべて処理済みでした。")
        return

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = [ex.submit(worker, i, m) for i, m in todo]
        for fut in tqdm(as_completed(futures), total=len(futures), desc="processing"):
            fut.result()

    print(f"完了: {len(todo)}件。保存先: {JSONL_PATH}")


# --------- 実行例 ----------
run_all(queries)

processing: 100%|██████████| 73/73 [00:07<00:00,  9.97it/s]

完了: 73件。保存先: out/many_shot/gpt-4o-2024-11-20/8/results.jsonl


# calculate QWK

In [146]:
import json
from json_repair import repair_json

results = []
errors = []

with open(JSONL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        try:
            entry = json.loads(line.strip())
        except json.JSONDecodeError:
            continue  # JSONとして読み込めない行はスキップ

        entry_id = entry['id']
        text = entry.get("text", "")

        try:
            good_json = repair_json(text)
            score = json.loads(good_json)['score']
            results.append({"essay_id": int(entry_id), "pred_score": score})
        except:
            results.append({"essay_id": int(entry_id), "pred_score": None})
            errors.append(f"ID {entry_id} のスコアが不正な形式です: {text}")


print(f"抽出結果: {len(results)}件、エラー: {len(errors)}件")

抽出結果: 73件、エラー: 0件


In [147]:
results_df = pl.DataFrame(results)
results_df

essay_id,pred_score
i64,i64
20785,30
20838,28
20734,38
20999,38
20929,28
…,…
21615,28
21607,45
21589,28


In [148]:
test_df

essay_set,essay_id,essay,score
i64,i64,str,i64
8,20838,"""Iv always loved laughter in my…",27
8,20728,""" Starting a story out with two…",44
8,20972,""" Falling in love, @CAPS…",32
8,21308,"""During the @DATE1 I decided to…",40
8,21298,""" One stormy winter night my g…",30
…,…,…,…
8,21029,""" the @CAPS1 @CAPS2 trip …",30
8,21532,""" ""@CAPS1 days are diamonds and…",50
8,20806,""" There are a couple things tha…",35


In [149]:
final_df = test_df.join(results_df, on='essay_id', how='left')
final_df

essay_set,essay_id,essay,score,pred_score
i64,i64,str,i64,i64
8,20838,"""Iv always loved laughter in my…",27,28
8,20728,""" Starting a story out with two…",44,36
8,20972,""" Falling in love, @CAPS…",32,28
8,21308,"""During the @DATE1 I decided to…",40,38
8,21298,""" One stormy winter night my g…",30,28
…,…,…,…,…
8,21029,""" the @CAPS1 @CAPS2 trip …",30,20
8,21532,""" ""@CAPS1 days are diamonds and…",50,45
8,20806,""" There are a couple things tha…",35,38


In [150]:
from sklearn.metrics import cohen_kappa_score

minscore, maxscore = get_score_range("ASAP", TARGET_PROMPT_ID)
qwk = cohen_kappa_score(
    final_df["pred_score"].to_numpy(),
    final_df["score"].to_numpy(),
    weights="quadratic",
    labels=list(range(minscore, maxscore + 1)),
)
print(f"QWK: {qwk:.4f}")

QWK: 0.6737


# Many-shot Performance

| Prompt.| QWK |
|----:|-------:|
| 1   | 0.5109 |
| 2   | 0.6056 |
| 3   | 0.5126 |
| 4   | 0.7858 |
| 5   | 0.7376 |
| 6   | 0.8018 |
| 7   | 0.6457 |
| 8   | 0.6737 |

avg = 0.6592